# Precipitation

This notebook separates precipitation analysis into two independent workflows:

- `daily`: GHCND `PRCP` daily precipitation for climatology, daily accumulations, monthly/annual summaries, and `daily_wet_spell` sequences.
- `hourly/subdaily`: only NOAA/NCEI sources that are actually verified at hourly or sub-hourly resolution for `hourly_storm_event` detection.

Key rule: a 6-hour dry gap is never applied to daily data. Daily data are segmented only as consecutive wet-day sequences.

Expected outputs:

- `data/processed/precipitation/station_availability/station_precipitation_availability.csv`
- `data/processed/precipitation/station_availability/station_precipitation_availability.gpkg`
- `data/processed/precipitation/daily/daily_precipitation_records.csv`
- `data/processed/precipitation/hourly/hourly_precipitation_records.csv`
- `data/processed/precipitation/events_daily/daily_wet_spell_events.csv`
- `data/processed/precipitation/events_hourly/hourly_storm_events.csv`


In [ ]:
import hashlib
import json
import logging
import os
import time
from datetime import date
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from IPython.display import display
from shapely import make_valid
from shapely.ops import unary_union

MPLCONFIGDIR = Path('/tmp/matplotlib')
MPLCONFIGDIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('MPLCONFIGDIR', str(MPLCONFIGDIR))

logger = logging.getLogger('precipitation')
if not logger.handlers:
    handler = logging.StreamHandler()
    handler.setFormatter(logging.Formatter('[%(levelname)s] %(message)s'))
    logger.addHandler(handler)
logger.setLevel(logging.INFO)


def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'pyproject.toml').exists() and (candidate / 'data').exists():
            return candidate
    return current


ROOT = find_project_root()
INVENTORY_PATH = ROOT / 'data' / 'temporal' / 'noaa' / '4308211.csv'
NYC_BOUNDARY_PATH = ROOT / 'data' / 'spatial' / 'vector' / 'nyc_borough_boundary' / 'nybb.geojson'

OUTPUT_ROOT = ROOT / 'data' / 'processed' / 'precipitation'
AVAILABILITY_DIR = OUTPUT_ROOT / 'station_availability'
DAILY_DIR = OUTPUT_ROOT / 'daily'
HOURLY_DIR = OUTPUT_ROOT / 'hourly'
DAILY_EVENTS_DIR = OUTPUT_ROOT / 'events_daily'
HOURLY_EVENTS_DIR = OUTPUT_ROOT / 'events_hourly'
RAW_CACHE_DIR = ROOT / 'data' / 'temporal' / 'noaa' / 'cache'

for folder in (AVAILABILITY_DIR, DAILY_DIR, HOURLY_DIR, DAILY_EVENTS_DIR, HOURLY_EVENTS_DIR, RAW_CACHE_DIR):
    folder.mkdir(parents=True, exist_ok=True)

NOAA_BASE_URL = 'https://www.ncei.noaa.gov/cdo-web/api/v2'
NOAA_TOKEN_ENV_NAMES = ('NOAA_CDO_TOKEN', 'NOAA_API_TOKEN')
START_DATE = '2010-01-01'
END_DATE = date.today().isoformat()
API_LIMIT = 1000
PROBE_LIMIT = 25
API_MAX_RETRIES = 5
API_RETRY_BACKOFF_SECONDS = 2.0
MIN_EVENT_MM = 1.1
DAILY_WET_GAP = pd.Timedelta(days=1)
HOURLY_DRY_GAP = pd.Timedelta(hours=6)
LOCAL_CRS = 'EPSG:2263'

# Keep these explicit and conservative. Discovery below may add more precipitation-like datatypes.
DAILY_CANDIDATES = [
    {'dataset_id': 'GHCND', 'datatype_id': 'PRCP', 'expected_resolution': 'daily', 'notes': 'GHCND daily precipitation'},
]

# NOAA CDO hourly/subdaily products can vary by station. Do not assume these exist.
# The notebook probes every station/dataset/datatype combination before selecting it.
HOURLY_CANDIDATES = [
    {'dataset_id': 'PRECIP_HLY', 'datatype_id': 'HPCP', 'expected_resolution': 'hourly', 'notes': 'Hourly Precipitation Data / HPCP'},
    {'dataset_id': 'LCD', 'datatype_id': 'HourlyPrecipitation', 'expected_resolution': 'hourly', 'notes': 'Local Climatological Data hourly precipitation'},
    {'dataset_id': 'LCD', 'datatype_id': 'HPCP', 'expected_resolution': 'hourly', 'notes': 'LCD HPCP fallback when available'},
]

DISCOVER_DATASET_IDS = ['PRECIP_HLY', 'LCD', 'GLOBAL_HOURLY']
PRECIP_DATATYPE_CACHE = {}
PROBE_CACHE = {}

TARGET_STATIONS = [
    'US1NJPS0012',
    'USC00283704',
    'USC00289187',
    'US1NJMN0010',
    'USC00066655',
    'US1NJMS0049',
    'US1NJPS0019',
    'USC00300961',
    'USC00287865',
    'USC00285503',
    'USC00302129',
    'USC00287079',
    'USC00306138',
]


def get_noaa_token() -> str | None:
    for name in NOAA_TOKEN_ENV_NAMES:
        token = os.getenv(name)
        if token:
            return token
    logger.warning('NOAA token missing. The notebook will use local cache if available; uncached API requests will return empty frames.')
    return None


def station_aliases(station_code: str, dataset_id: str | None = None) -> list[str]:
    """Return station-id aliases commonly used by NOAA CDO products."""
    prefixes = ['GHCND', 'COOP', 'LCD', 'WBAN']
    aliases = []
    if dataset_id == 'GHCND':
        aliases.append(f'GHCND:{station_code}')
    elif dataset_id == 'PRECIP_HLY':
        aliases.extend([f'COOP:{station_code}', f'GHCND:{station_code}'])
    elif dataset_id == 'LCD':
        aliases.extend([f'LCD:{station_code}', f'COOP:{station_code}', f'GHCND:{station_code}'])
    elif dataset_id == 'GLOBAL_HOURLY':
        aliases.extend([f'GHCND:{station_code}', f'COOP:{station_code}'])
    aliases.append(station_code)
    aliases.extend([f'{prefix}:{station_code}' for prefix in prefixes])
    seen = set()
    out = []
    for alias in aliases:
        if alias not in seen:
            out.append(alias)
            seen.add(alias)
    return out


def request_cdo_json(endpoint: str, token: str, params: dict | None = None) -> dict:
    url = f'{NOAA_BASE_URL}/{endpoint.lstrip("/")}'
    headers = {'token': token}
    params = dict(params or {})
    for attempt in range(API_MAX_RETRIES + 1):
        response = requests.get(url, headers=headers, params=params, timeout=60)
        if response.status_code in {429, 500, 502, 503, 504} and attempt < API_MAX_RETRIES:
            wait_seconds = API_RETRY_BACKOFF_SECONDS * (2 ** attempt)
            logger.warning('NOAA retry status=%s endpoint=%s attempt=%s/%s sleep=%.1fs', response.status_code, endpoint, attempt + 1, API_MAX_RETRIES + 1, wait_seconds)
            time.sleep(wait_seconds)
            continue
        if response.status_code >= 400:
            body = response.text.strip().replace('\n', ' ')[:500]
            raise RuntimeError(f'Request failed for {url}: {response.status_code} {response.reason}. {body}')
        try:
            return response.json()
        except ValueError as exc:
            body = response.text.strip().replace('\n', ' ')[:500]
            raise RuntimeError(f'Invalid JSON from {url}: {body}') from exc
    raise RuntimeError(f'Unable to fetch {url}')


def build_cache_path(kind: str, dataset_id: str, datatype_id: str, station_id: str, start_date: str, end_date: str) -> Path:
    key = f'{kind}|{dataset_id}|{datatype_id}|{station_id}|{start_date}|{end_date}'
    digest = hashlib.sha1(key.encode('utf-8')).hexdigest()[:16]
    safe_dataset = dataset_id.replace(':', '_')
    safe_datatype = datatype_id.replace(':', '_')
    return RAW_CACHE_DIR / f'precip_{kind}_{safe_dataset}_{safe_datatype}_{digest}.csv'


def fetch_cdo_frame(token: str | None, dataset_id: str, datatype_id: str, station_id: str, start_date: str, end_date: str, limit: int, kind: str, force_refresh: bool = False) -> pd.DataFrame:
    cache_path = build_cache_path(kind, dataset_id, datatype_id, station_id, start_date, end_date)
    if cache_path.exists() and not force_refresh:
        frame = pd.read_csv(cache_path, low_memory=False)
        frame.attrs['record_count'] = len(frame)
        frame.attrs['cache_path'] = str(cache_path)
        return frame
    if not token:
        empty = pd.DataFrame()
        empty.attrs['record_count'] = 0
        return empty

    rows = []
    offset = 0
    total_count = None
    while True:
        payload = request_cdo_json(
            'data',
            token,
            params={
                'datasetid': dataset_id,
                'datatypeid': datatype_id,
                'stationid': station_id,
                'startdate': start_date,
                'enddate': end_date,
                'units': 'metric',
                'limit': limit,
                'offset': offset,
            },
        )
        batch = payload.get('results', [])
        if total_count is None:
            total_count = payload.get('metadata', {}).get('resultset', {}).get('count')
        if not batch:
            break
        rows.extend(batch)
        if len(batch) < limit:
            break
        offset += limit

    frame = pd.DataFrame.from_records(rows)
    if not frame.empty:
        frame.to_csv(cache_path, index=False)
    frame.attrs['record_count'] = int(total_count) if total_count is not None else len(frame)
    frame.attrs['cache_path'] = str(cache_path)
    return frame


def prepare_cdo_frame(frame: pd.DataFrame, dataset_id: str | None = None, datatype_id: str | None = None) -> pd.DataFrame:
    out = frame.copy()
    if out.empty:
        return pd.DataFrame(columns=['timestamp', 'source_station_id', 'precip_mm'])
    rename_map = {}
    if 'date' in out.columns and 'timestamp' not in out.columns:
        rename_map['date'] = 'timestamp'
    if 'station' in out.columns and 'source_station_id' not in out.columns:
        rename_map['station'] = 'source_station_id'
    if 'value' in out.columns and 'precip_mm' not in out.columns:
        rename_map['value'] = 'precip_mm'
    out = out.rename(columns=rename_map)
    if 'timestamp' not in out.columns or 'precip_mm' not in out.columns:
        raise ValueError(f'NOAA response lacks required columns: {sorted(out.columns)}')
    out['timestamp'] = pd.to_datetime(out['timestamp'], errors='coerce')

    # CDO usually returns metric precipitation in millimeters for PRCP/HPCP/HourlyPrecipitation.
    # Trace values can appear as text; encode traces as 0.0 mm to preserve timestamps but avoid false storms.
    raw = out['precip_mm'].astype('string').str.strip()
    trace_mask = raw.str.upper().isin({'T', 'TRACE'})
    raw = raw.str.replace(r'[^0-9eE+\-.]', '', regex=True)
    out['precip_mm'] = pd.to_numeric(raw, errors='coerce')
    out.loc[trace_mask, 'precip_mm'] = 0.0
    out = out.dropna(subset=['timestamp', 'precip_mm']).copy()
    out = out.loc[out['precip_mm'] >= 0].copy()
    if 'source_station_id' not in out.columns:
        out['source_station_id'] = pd.NA
    out['source_station_id'] = out['source_station_id'].astype('string')
    return out[['timestamp', 'source_station_id', 'precip_mm']].copy()


def infer_temporal_resolution(timestamps, dataset_id: str | None = None, datatype_id: str | None = None) -> str:
    ts = pd.Series(pd.to_datetime(timestamps, errors='coerce')).dropna().sort_values().drop_duplicates()
    if len(ts) < 2:
        if dataset_id == 'GHCND' and datatype_id == 'PRCP':
            return 'daily'
        if datatype_id and str(datatype_id).upper() in {'HPCP', 'HOURLYPRECIPITATION'}:
            return 'hourly'
        return 'unknown'
    diffs = ts.diff().dropna()
    diffs = diffs[diffs > pd.Timedelta(0)]
    if diffs.empty:
        return 'unknown'
    min_delta = diffs.min()
    if min_delta < pd.Timedelta(hours=1):
        return 'subhourly'
    if min_delta < pd.Timedelta(days=1):
        return 'hourly'
    return 'daily'


def discover_precip_datatypes(dataset_id: str, token: str | None) -> list[dict]:
    """Discover precipitation-related datatypes for a NOAA CDO dataset.

    This does not mark availability. It only expands the candidate list to probe by station.
    """
    if dataset_id in PRECIP_DATATYPE_CACHE:
        return PRECIP_DATATYPE_CACHE[dataset_id]
    if not token:
        PRECIP_DATATYPE_CACHE[dataset_id] = []
        return []
    try:
        payload = request_cdo_json('datatypes', token, params={'datasetid': dataset_id, 'limit': 1000})
    except Exception as exc:
        logger.info('Could not discover datatypes for dataset=%s: %s', dataset_id, exc)
        PRECIP_DATATYPE_CACHE[dataset_id] = []
        return []
    rows = payload.get('results', [])
    candidates = []
    for row in rows:
        datatype_id = str(row.get('id', ''))
        name = str(row.get('name', ''))
        text = f'{datatype_id} {name}'.lower()
        if not any(key in text for key in ['precip', 'rain', 'hpcp', 'prcp']):
            continue
        if 'snow' in text and not any(key in text for key in ['precip', 'hpcp', 'prcp']):
            continue
        expected = 'hourly' if any(key in text for key in ['hourly', 'hpcp']) else 'unknown'
        candidates.append({'dataset_id': dataset_id, 'datatype_id': datatype_id, 'expected_resolution': expected, 'notes': f'discovered: {name}'})
    PRECIP_DATATYPE_CACHE[dataset_id] = candidates
    return candidates


def unique_candidates(candidates: list[dict]) -> list[dict]:
    seen = set()
    out = []
    for item in candidates:
        key = (item['dataset_id'], item['datatype_id'])
        if key in seen:
            continue
        out.append(item)
        seen.add(key)
    return out


def hourly_candidate_table(token: str | None) -> list[dict]:
    discovered = []
    for dataset_id in DISCOVER_DATASET_IDS:
        discovered.extend(discover_precip_datatypes(dataset_id, token))
    return unique_candidates(HOURLY_CANDIDATES + discovered)


def probe_station_source(token: str | None, dataset_id: str, datatype_id: str, anchor_station_id: str, station_name: str, required_resolution: str, notes: str = '') -> dict | None:
    aliases = station_aliases(anchor_station_id, dataset_id=dataset_id)
    cache_key = (dataset_id, datatype_id, anchor_station_id, required_resolution, tuple(aliases))
    if cache_key in PROBE_CACHE:
        return PROBE_CACHE[cache_key]
    if not token:
        PROBE_CACHE[cache_key] = None
        return None

    errors = []
    for alias in aliases:
        try:
            raw = fetch_cdo_frame(token=token, dataset_id=dataset_id, datatype_id=datatype_id, station_id=alias, start_date=START_DATE, end_date=END_DATE, limit=PROBE_LIMIT, kind='probe')
        except Exception as exc:
            errors.append(f'{alias}: {exc}')
            continue
        if raw.empty:
            continue
        try:
            prepared = prepare_cdo_frame(raw, dataset_id=dataset_id, datatype_id=datatype_id)
        except Exception as exc:
            errors.append(f'{alias}: parse_error={exc}')
            continue
        if prepared.empty:
            continue
        resolution = infer_temporal_resolution(prepared['timestamp'], dataset_id=dataset_id, datatype_id=datatype_id)
        if required_resolution == 'daily' and resolution != 'daily':
            continue
        if required_resolution == 'hourly' and resolution not in {'hourly', 'subhourly'}:
            continue
        result = {
            'station_id': anchor_station_id,
            'station_name': station_name,
            'source_dataset': dataset_id,
            'data_type': datatype_id,
            'source_station_id': str(prepared['source_station_id'].dropna().iloc[0]) if prepared['source_station_id'].notna().any() else alias,
            'source_alias_used': alias,
            'temporal_resolution': resolution,
            'record_count_probe': int(raw.attrs.get('record_count', len(prepared))),
            'sample_start': prepared['timestamp'].min(),
            'sample_end': prepared['timestamp'].max(),
            'candidate_notes': notes,
        }
        PROBE_CACHE[cache_key] = result
        return result
    if errors:
        logger.info('Probe failed for station=%s dataset=%s datatype=%s examples=%s', anchor_station_id, dataset_id, datatype_id, ' | '.join(errors[:3]))
    PROBE_CACHE[cache_key] = None
    return None


def select_precip_source(token: str | None, station_row: pd.Series, kind: str) -> tuple[dict | None, list[str]]:
    station_id = str(station_row['station_id'])
    station_name = str(station_row['station_name'])
    checked = []
    if kind == 'daily':
        candidates = DAILY_CANDIDATES
        required_resolution = 'daily'
    elif kind == 'hourly':
        candidates = hourly_candidate_table(token)
        required_resolution = 'hourly'
    else:
        raise ValueError(f'Unknown kind: {kind}')

    for candidate in candidates:
        dataset_id = candidate['dataset_id']
        datatype_id = candidate['datatype_id']
        checked.append(f'{dataset_id}:{datatype_id}')
        probe = probe_station_source(
            token=token,
            dataset_id=dataset_id,
            datatype_id=datatype_id,
            anchor_station_id=station_id,
            station_name=station_name,
            required_resolution=required_resolution,
            notes=candidate.get('notes', ''),
        )
        if probe is not None:
            probe['source_kind'] = kind
            probe['checked_candidates'] = checked.copy()
            return probe, checked
    return None, checked


def audit_station_availability(station_row: pd.Series, token: str | None) -> dict:
    daily, daily_checked = select_precip_source(token, station_row, 'daily')
    hourly, hourly_checked = select_precip_source(token, station_row, 'hourly')
    selected = hourly or daily
    if selected is None:
        station_type = 'no_precip_found'
        selected_kind = 'none'
    elif selected['temporal_resolution'] == 'subhourly':
        station_type = 'subhourly_available'
        selected_kind = 'hourly'
    elif selected['temporal_resolution'] == 'hourly':
        station_type = 'hourly_available'
        selected_kind = 'hourly'
    else:
        station_type = 'daily_only'
        selected_kind = 'daily'

    notes = []
    notes.append(f'checked_daily={"|".join(daily_checked) if daily_checked else "none"}')
    notes.append(f'checked_hourly={"|".join(hourly_checked) if hourly_checked else "none"}')
    if daily:
        notes.append(f"daily_found={daily['source_dataset']}:{daily['data_type']}:{daily['source_station_id']}")
    else:
        notes.append('daily_not_found')
    if hourly:
        notes.append(f"hourly_found={hourly['source_dataset']}:{hourly['data_type']}:{hourly['source_station_id']}")
    else:
        notes.append('hourly_not_found')
    if selected is None:
        notes.append('excluded_no_precip')
    if str(station_row['station_id']).startswith('US1'):
        notes.append('US1_candidate_probably_CoCoRaHS_daily_manual')

    return {
        'station_id': str(station_row['station_id']),
        'station_name': str(station_row['station_name']),
        'station_type_prelim': str(station_row['station_type_prelim']),
        'station_type': station_type,
        'latitude': float(station_row['latitude']),
        'longitude': float(station_row['longitude']),
        'selected_source_kind': selected_kind,
        'source_dataset': selected['source_dataset'] if selected else pd.NA,
        'data_type': selected['data_type'] if selected else pd.NA,
        'temporal_resolution': selected['temporal_resolution'] if selected else pd.NA,
        'source_station_id': selected['source_station_id'] if selected else pd.NA,
        'source_alias_used': selected['source_alias_used'] if selected else pd.NA,
        'record_count': 0,
        'start_date': pd.NA,
        'end_date': pd.NA,
        'daily_source_dataset': daily['source_dataset'] if daily else pd.NA,
        'daily_data_type': daily['data_type'] if daily else pd.NA,
        'daily_source_station_id': daily['source_station_id'] if daily else pd.NA,
        'daily_probe_resolution': daily['temporal_resolution'] if daily else pd.NA,
        'daily_record_count_probe': daily['record_count_probe'] if daily else 0,
        'hourly_source_dataset': hourly['source_dataset'] if hourly else pd.NA,
        'hourly_data_type': hourly['data_type'] if hourly else pd.NA,
        'hourly_source_station_id': hourly['source_station_id'] if hourly else pd.NA,
        'hourly_probe_resolution': hourly['temporal_resolution'] if hourly else pd.NA,
        'hourly_record_count_probe': hourly['record_count_probe'] if hourly else 0,
        'has_daily_prcp': bool(daily),
        'has_hourly_prcp': bool(hourly),
        'has_subhourly_prcp': bool(hourly and hourly['temporal_resolution'] == 'subhourly'),
        'selected_for_daily_analysis': bool(daily),
        'selected_for_event_analysis': bool(hourly),
        'daily_checked_candidates': '|'.join(daily_checked),
        'hourly_checked_candidates': '|'.join(hourly_checked),
        'notes': '; '.join(notes),
        'daily_sample_start': daily['sample_start'] if daily else pd.NA,
        'daily_sample_end': daily['sample_end'] if daily else pd.NA,
        'hourly_sample_start': hourly['sample_start'] if hourly else pd.NA,
        'hourly_sample_end': hourly['sample_end'] if hourly else pd.NA,
    }


def load_station_inventory(target_stations: list[str]) -> pd.DataFrame:
    frame = pd.read_csv(INVENTORY_PATH, low_memory=False)
    required = {'STATION', 'NAME', 'LATITUDE', 'LONGITUDE'}
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f'NOAA inventory lacks expected columns: {sorted(missing)}')
    frame = frame.drop_duplicates('STATION').copy()
    frame = frame.rename(columns={'STATION': 'station_id', 'NAME': 'station_name', 'LATITUDE': 'latitude', 'LONGITUDE': 'longitude'})
    frame['station_id'] = frame['station_id'].astype('string')
    frame['station_name'] = frame['station_name'].astype('string')
    frame['latitude'] = pd.to_numeric(frame['latitude'], errors='coerce')
    frame['longitude'] = pd.to_numeric(frame['longitude'], errors='coerce')
    frame = frame.dropna(subset=['station_id', 'latitude', 'longitude']).copy()
    frame['station_type_prelim'] = np.where(frame['station_id'].str.startswith('US1'), 'probable_daily_only', 'daily_available_hourly_possible')
    frame = frame.loc[frame['station_id'].isin(target_stations)].copy()
    missing_targets = sorted(set(target_stations) - set(frame['station_id']))
    if missing_targets:
        logger.warning('Station ids not found in inventory: %s', ', '.join(missing_targets))
    return frame.sort_values('station_id').reset_index(drop=True)


def load_nyc_boundary() -> tuple[gpd.GeoDataFrame, object]:
    boroughs = gpd.read_file(NYC_BOUNDARY_PATH)
    boroughs = boroughs.copy()
    boroughs['geometry'] = boroughs.geometry.apply(make_valid)
    if boroughs.crs is None:
        boroughs = boroughs.set_crs('EPSG:4326')
    return boroughs, unary_union(boroughs.geometry)


def make_station_gdf(frame: pd.DataFrame, crs: str = 'EPSG:4326') -> gpd.GeoDataFrame:
    return gpd.GeoDataFrame(frame.copy(), geometry=gpd.points_from_xy(frame['longitude'], frame['latitude']), crs=crs)


def set_map_extent(ax, geom, pad: float = 0.05) -> None:
    minx, miny, maxx, maxy = geom.bounds
    dx = (maxx - minx) * pad
    dy = (maxy - miny) * pad
    ax.set_xlim(minx - dx, maxx + dx)
    ax.set_ylim(miny - dy, maxy + dy)


def validate_series(frame: pd.DataFrame, label: str) -> pd.DataFrame:
    if frame.empty:
        return frame
    required = {'station_id', 'timestamp', 'precip_mm'}
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f'{label}: missing required columns {sorted(missing)}')
    frame = frame.dropna(subset=['station_id', 'timestamp', 'precip_mm']).copy()
    dup_mask = frame.duplicated(subset=['station_id', 'timestamp'])
    if dup_mask.any():
        logger.warning('%s: removed %s duplicate station-timestamps', label, int(dup_mask.sum()))
        frame = frame.loc[~dup_mask].copy()
    bad = frame['precip_mm'] < 0
    if bad.any():
        logger.warning('%s: removed %s negative precipitation values', label, int(bad.sum()))
        frame = frame.loc[~bad].copy()
    extreme = frame['precip_mm'] > 500
    if extreme.any():
        logger.warning('%s: %s records exceed 500 mm. They are retained but should be checked.', label, int(extreme.sum()))
    frame = frame.sort_values(['station_id', 'timestamp']).reset_index(drop=True)
    return frame


def finalize_record_frame(frame: pd.DataFrame, anchor_station_id: str, station_name: str, source_dataset: str, data_type: str, source_station_id: str, analysis_kind: str) -> pd.DataFrame:
    out = prepare_cdo_frame(frame, dataset_id=source_dataset, datatype_id=data_type)
    columns = ['station_id', 'station_name', 'source_station_id', 'source_dataset', 'data_type', 'analysis_kind', 'temporal_resolution', 'timestamp', 'precip_mm', 'year', 'month', 'day']
    if out.empty:
        return pd.DataFrame(columns=columns)
    out['station_id'] = anchor_station_id
    out['station_name'] = station_name
    out['source_station_id'] = out['source_station_id'].fillna(source_station_id)
    out['source_dataset'] = source_dataset
    out['data_type'] = data_type
    out['analysis_kind'] = analysis_kind
    resolution = infer_temporal_resolution(out['timestamp'], dataset_id=source_dataset, datatype_id=data_type)
    out['temporal_resolution'] = resolution
    if analysis_kind == 'daily' and resolution != 'daily':
        raise ValueError(f'{anchor_station_id}: daily pipeline received non-daily data from {source_dataset}:{data_type}')
    if analysis_kind == 'hourly' and resolution not in {'hourly', 'subhourly'}:
        raise ValueError(f'{anchor_station_id}: hourly pipeline received {resolution} data from {source_dataset}:{data_type}')
    out = out.drop_duplicates(subset=['station_id', 'timestamp']).copy()
    out = out.sort_values(['station_id', 'timestamp']).reset_index(drop=True)
    out['year'] = out['timestamp'].dt.year
    out['month'] = out['timestamp'].dt.month
    out['day'] = out['timestamp'].dt.day
    return out[columns]


def build_precip_event_summary(records: pd.DataFrame, event_label: str, gap: pd.Timedelta, min_total_mm: float) -> pd.DataFrame:
    columns = ['station_id', 'station_name', 'event_id', 'event_uid', 'event_kind', 'start', 'end', 'duration_hours', 'precip_mm', 'mean_intensity_mm_per_hr', 'n_records', 'wet_records', 'source_dataset', 'data_type', 'temporal_resolution']
    if records.empty:
        return pd.DataFrame(columns=columns)
    if event_label == 'hourly_storm_event':
        bad_res = set(records['temporal_resolution'].dropna()) - {'hourly', 'subhourly'}
        if bad_res:
            raise ValueError(f'Hourly storm event detection received invalid resolutions: {sorted(bad_res)}')
    if event_label == 'daily_wet_spell':
        bad_res = set(records['temporal_resolution'].dropna()) - {'daily'}
        if bad_res:
            raise ValueError(f'Daily wet-spell detection received invalid resolutions: {sorted(bad_res)}')

    summaries = []
    group_cols = ['station_id', 'station_name', 'source_dataset', 'data_type', 'temporal_resolution']
    for (station_id, station_name, source_dataset, data_type, temporal_resolution), group in records.groupby(group_cols, dropna=False, sort=False):
        group = group.sort_values('timestamp').copy()
        wet = group.loc[group['precip_mm'] > 0, ['timestamp', 'precip_mm']].copy()
        if wet.empty:
            continue
        wet['time_diff'] = wet['timestamp'].diff()
        wet['event_id'] = (wet['time_diff'].isna() | (wet['time_diff'] > gap)).cumsum().astype(int)
        for event_id, wet_group in wet.groupby('event_id', sort=True):
            start = wet_group['timestamp'].min()
            end = wet_group['timestamp'].max()
            event_frame = group.loc[(group['timestamp'] >= start) & (group['timestamp'] <= end)].copy()
            if event_frame.empty:
                continue
            total_mm = float(event_frame['precip_mm'].sum())
            if total_mm < min_total_mm:
                continue
            step = event_frame['timestamp'].sort_values().diff().dropna()
            sample_step = step[step > pd.Timedelta(0)].min() if not step.empty else pd.NaT
            if pd.isna(sample_step):
                sample_step = pd.Timedelta(days=1) if event_label == 'daily_wet_spell' else pd.Timedelta(hours=1)
            duration_hours = max(((end - start) + sample_step).total_seconds() / 3600.0, sample_step.total_seconds() / 3600.0)
            summaries.append({
                'station_id': station_id,
                'station_name': station_name,
                'event_id': int(event_id),
                'event_uid': f'{station_id}_{event_label}_{int(event_id):04d}',
                'event_kind': event_label,
                'start': start,
                'end': end,
                'duration_hours': duration_hours,
                'precip_mm': total_mm,
                'mean_intensity_mm_per_hr': total_mm / duration_hours if duration_hours > 0 else np.nan,
                'n_records': int(len(event_frame)),
                'wet_records': int(len(wet_group)),
                'source_dataset': source_dataset,
                'data_type': data_type,
                'temporal_resolution': temporal_resolution,
            })
    if not summaries:
        return pd.DataFrame(columns=columns)
    return pd.DataFrame.from_records(summaries).sort_values(['station_id', 'start', 'event_id'], kind='stable').reset_index(drop=True)[columns]


def summarize_station_records(frame: pd.DataFrame) -> pd.DataFrame:
    columns = ['station_id', 'station_name', 'source_dataset', 'data_type', 'analysis_kind', 'temporal_resolution', 'start', 'end', 'record_count', 'wet_records', 'missing_intervals_estimate', 'total_precip_mm', 'mean_precip_mm']
    if frame.empty:
        return pd.DataFrame(columns=columns)
    def missing_intervals(group):
        ts = group['timestamp'].sort_values().drop_duplicates()
        if len(ts) < 3:
            return 0
        diffs = ts.diff().dropna()
        step = diffs[diffs > pd.Timedelta(0)].mode()
        if step.empty:
            step = diffs[diffs > pd.Timedelta(0)].min()
        else:
            step = step.iloc[0]
        if pd.isna(step) or step <= pd.Timedelta(0):
            return 0
        expected = int(((ts.max() - ts.min()) / step)) + 1
        return max(expected - len(ts), 0)
    summary = (
        frame.groupby(['station_id', 'station_name', 'source_dataset', 'data_type', 'analysis_kind', 'temporal_resolution'], dropna=False)
        .agg(
            start=('timestamp', 'min'),
            end=('timestamp', 'max'),
            record_count=('timestamp', 'size'),
            wet_records=('precip_mm', lambda s: int((s > 0).sum())),
            total_precip_mm=('precip_mm', 'sum'),
            mean_precip_mm=('precip_mm', 'mean'),
        )
        .reset_index()
    )
    miss = frame.groupby(['station_id', 'station_name', 'source_dataset', 'data_type', 'analysis_kind', 'temporal_resolution'], dropna=False).apply(missing_intervals).rename('missing_intervals_estimate').reset_index()
    summary = summary.merge(miss, on=['station_id', 'station_name', 'source_dataset', 'data_type', 'analysis_kind', 'temporal_resolution'], how='left')
    return summary[columns]


def build_final_availability_table(probe_frame: pd.DataFrame, daily_records: pd.DataFrame, hourly_records: pd.DataFrame) -> pd.DataFrame:
    frame = probe_frame.copy()
    daily_summary = summarize_station_records(daily_records).rename(columns={
        'start': 'daily_start_date', 'end': 'daily_end_date', 'record_count': 'daily_record_count', 'wet_records': 'daily_wet_records', 'missing_intervals_estimate': 'daily_missing_intervals_estimate', 'total_precip_mm': 'daily_total_precip_mm', 'mean_precip_mm': 'daily_mean_precip_mm',
    })
    hourly_summary = summarize_station_records(hourly_records).rename(columns={
        'start': 'hourly_start_date', 'end': 'hourly_end_date', 'record_count': 'hourly_record_count', 'wet_records': 'hourly_wet_records', 'missing_intervals_estimate': 'hourly_missing_intervals_estimate', 'total_precip_mm': 'hourly_total_precip_mm', 'mean_precip_mm': 'hourly_mean_precip_mm',
    })
    daily_summary = daily_summary.drop(columns=[c for c in ['source_dataset', 'data_type', 'analysis_kind', 'temporal_resolution'] if c in daily_summary.columns])
    hourly_summary = hourly_summary.drop(columns=[c for c in ['source_dataset', 'data_type', 'analysis_kind', 'temporal_resolution'] if c in hourly_summary.columns])
    frame = frame.merge(daily_summary, on=['station_id', 'station_name'], how='left')
    frame = frame.merge(hourly_summary, on=['station_id', 'station_name'], how='left')

    frame['has_daily_prcp'] = frame['daily_record_count'].fillna(0).astype(int) > 0
    frame['has_hourly_prcp'] = frame['hourly_record_count'].fillna(0).astype(int) > 0
    frame['has_subhourly_prcp'] = frame['has_hourly_prcp'] & frame['hourly_probe_resolution'].eq('subhourly')
    frame['selected_for_daily_analysis'] = frame['has_daily_prcp']
    frame['selected_for_event_analysis'] = frame['has_hourly_prcp']

    frame['selected_source_kind'] = np.where(frame['has_hourly_prcp'], 'hourly', np.where(frame['has_daily_prcp'], 'daily', 'none'))
    frame['station_type'] = np.where(frame['has_subhourly_prcp'], 'subhourly_available', np.where(frame['has_hourly_prcp'], 'hourly_available', np.where(frame['has_daily_prcp'], 'daily_only', 'no_precip_found')))
    mask_hourly = frame['selected_source_kind'].eq('hourly')
    mask_daily = frame['selected_source_kind'].eq('daily')
    frame['source_dataset'] = np.where(mask_hourly, frame['hourly_source_dataset'], np.where(mask_daily, frame['daily_source_dataset'], pd.NA))
    frame['data_type'] = np.where(mask_hourly, frame['hourly_data_type'], np.where(mask_daily, frame['daily_data_type'], pd.NA))
    frame['temporal_resolution'] = np.where(mask_hourly, frame['hourly_probe_resolution'], np.where(mask_daily, frame['daily_probe_resolution'], pd.NA))
    frame['source_station_id'] = np.where(mask_hourly, frame['hourly_source_station_id'], np.where(mask_daily, frame['daily_source_station_id'], pd.NA))
    frame['record_count'] = np.where(mask_hourly, frame['hourly_record_count'], np.where(mask_daily, frame['daily_record_count'], 0))
    frame['record_count'] = pd.to_numeric(frame['record_count'], errors='coerce').fillna(0).astype('int64')
    frame['start_date'] = np.where(mask_hourly, frame['hourly_start_date'], np.where(mask_daily, frame['daily_start_date'], pd.NA))
    frame['end_date'] = np.where(mask_hourly, frame['hourly_end_date'], np.where(mask_daily, frame['daily_end_date'], pd.NA))

    for column in ['start_date', 'end_date', 'daily_start_date', 'daily_end_date', 'hourly_start_date', 'hourly_end_date']:
        if column in frame.columns:
            frame[column] = pd.to_datetime(frame[column], errors='coerce').dt.strftime('%Y-%m-%dT%H:%M:%S')

    ordered = [
        'station_id', 'station_name', 'station_type_prelim', 'station_type', 'latitude', 'longitude',
        'selected_source_kind', 'source_dataset', 'data_type', 'temporal_resolution', 'source_station_id', 'source_alias_used',
        'start_date', 'end_date', 'record_count', 'has_daily_prcp', 'has_hourly_prcp', 'has_subhourly_prcp',
        'selected_for_daily_analysis', 'selected_for_event_analysis',
        'daily_source_dataset', 'daily_data_type', 'daily_source_station_id', 'daily_probe_resolution', 'daily_record_count_probe', 'daily_start_date', 'daily_end_date', 'daily_record_count', 'daily_wet_records', 'daily_missing_intervals_estimate', 'daily_total_precip_mm', 'daily_mean_precip_mm',
        'hourly_source_dataset', 'hourly_data_type', 'hourly_source_station_id', 'hourly_probe_resolution', 'hourly_record_count_probe', 'hourly_start_date', 'hourly_end_date', 'hourly_record_count', 'hourly_wet_records', 'hourly_missing_intervals_estimate', 'hourly_total_precip_mm', 'hourly_mean_precip_mm',
        'daily_checked_candidates', 'hourly_checked_candidates', 'notes',
    ]
    for column in ordered:
        if column not in frame.columns:
            frame[column] = pd.NA
    return frame[ordered].sort_values('station_id').reset_index(drop=True)


## 1. Station availability audit

`US1*` stations are treated as probable CoCoRaHS/manual daily-only candidates. `USC*` stations are treated only as *hourly possible*, not hourly available.

The audit probes station/dataset/datatype combinations before assigning a station to event-scale analysis. In particular, it checks GHCND daily precipitation separately from hourly/subdaily candidate products such as NOAA hourly precipitation and LCD precipitation fields. A station is selected for storm-event analysis only when the returned timestamps verify hourly or sub-hourly resolution.


In [ ]:
token = get_noaa_token()
nyc_boroughs, nyc_union = load_nyc_boundary()
station_inventory = load_station_inventory(TARGET_STATIONS)

if station_inventory.empty:
    raise ValueError('No target stations were found in the NOAA inventory.')

candidate_hourly = pd.DataFrame(hourly_candidate_table(token))
print('Hourly/subdaily candidate datatypes that will be probed:')
display(candidate_hourly)

availability_probe = pd.DataFrame([audit_station_availability(row, token) for row in station_inventory.to_dict('records')])
availability_points = make_station_gdf(availability_probe, crs='EPSG:4326')
availability_points_local = availability_points.to_crs(LOCAL_CRS)

# Save preliminary availability immediately, even before full downloads.
availability_probe.to_csv(AVAILABILITY_DIR / 'station_precipitation_availability_probe.csv', index=False)
availability_points.to_file(AVAILABILITY_DIR / 'station_precipitation_availability_probe.gpkg', driver='GPKG', layer='station_precipitation_availability_probe')
availability_points_local.to_file(AVAILABILITY_DIR / 'station_precipitation_availability_probe_projected.gpkg', driver='GPKG', layer='station_precipitation_availability_probe_projected')

print(f'Stations evaluated: {len(availability_probe):,}')
print(f'Daily verified: {int(availability_probe["selected_for_daily_analysis"].sum()):,}')
print(f'Hourly verified: {int(availability_probe["selected_for_event_analysis"].sum()):,}')
print(f'Subhourly verified: {int(availability_probe["has_subhourly_prcp"].sum()):,}')

display(availability_probe[['station_id', 'station_type_prelim', 'station_type', 'selected_source_kind', 'source_dataset', 'data_type', 'temporal_resolution', 'selected_for_daily_analysis', 'selected_for_event_analysis', 'notes']])

fig, ax = plt.subplots(figsize=(10, 10))
boroughs_plot = nyc_boroughs.to_crs(availability_points.crs)
boroughs_plot.boundary.plot(ax=ax, color='black', linewidth=1)
colors = {'daily_only': '#2563eb', 'hourly_available': '#d97706', 'subhourly_available': '#dc2626', 'no_precip_found': '#6b7280'}
for station_type, group in availability_points.groupby('station_type'):
    group.plot(ax=ax, color=colors.get(station_type, '#6b7280'), markersize=40, label=station_type)
set_map_extent(ax, boroughs_plot.unary_union, pad=0.05)
ax.set_title('Precipitation station availability in NYC area', fontsize=13)
ax.set_axis_off()
ax.legend(frameon=False, loc='lower left')
plt.show()


## 2. Daily pipeline

This path uses GHCND `PRCP` daily precipitation. It is appropriate for climatology, daily accumulations, monthly/annual statistics, and consecutive wet-day sequences.

It does **not** use a 6-hour storm gap. Daily events are labeled `daily_wet_spell`.


In [ ]:
def fetch_selected_series(selection: pd.DataFrame, kind: str, token: str | None) -> pd.DataFrame:
    frames = []
    for row in selection.to_dict('records'):
        source_dataset = row[f'{kind}_source_dataset']
        data_type = row[f'{kind}_data_type']
        source_station_id = row[f'{kind}_source_station_id']
        if pd.isna(source_dataset) or pd.isna(data_type) or pd.isna(source_station_id):
            continue
        try:
            raw = fetch_cdo_frame(token=token, dataset_id=str(source_dataset), datatype_id=str(data_type), station_id=str(source_station_id), start_date=START_DATE, end_date=END_DATE, limit=API_LIMIT, kind=kind)
        except Exception as exc:
            logger.warning('%s: failed to fetch station=%s dataset=%s datatype=%s: %s', kind, row['station_id'], source_dataset, data_type, exc)
            continue
        if raw.empty:
            logger.warning('%s: no data fetched for station=%s dataset=%s datatype=%s', kind, row['station_id'], source_dataset, data_type)
            continue
        try:
            frame = finalize_record_frame(raw, anchor_station_id=row['station_id'], station_name=row['station_name'], source_dataset=str(source_dataset), data_type=str(data_type), source_station_id=str(source_station_id), analysis_kind=kind)
        except Exception as exc:
            logger.warning('%s: fetched data failed validation for station=%s dataset=%s datatype=%s: %s', kind, row['station_id'], source_dataset, data_type, exc)
            continue
        frames.append(frame)
    if not frames:
        return pd.DataFrame(columns=['station_id', 'station_name', 'source_station_id', 'source_dataset', 'data_type', 'analysis_kind', 'temporal_resolution', 'timestamp', 'precip_mm', 'year', 'month', 'day'])
    return validate_series(pd.concat(frames, ignore_index=True), f'{kind}_records')


daily_selection = availability_probe.loc[availability_probe['selected_for_daily_analysis']].copy()
daily_records = fetch_selected_series(daily_selection, 'daily', token) if not daily_selection.empty else pd.DataFrame(columns=['station_id', 'station_name', 'source_station_id', 'source_dataset', 'data_type', 'analysis_kind', 'temporal_resolution', 'timestamp', 'precip_mm', 'year', 'month', 'day'])

daily_records.to_csv(DAILY_DIR / 'daily_precipitation_records.csv', index=False)
daily_station_summary = summarize_station_records(daily_records)
daily_station_summary.to_csv(DAILY_DIR / 'daily_station_summary.csv', index=False)

if not daily_records.empty:
    daily_monthly_summary = (
        daily_records.assign(period=daily_records['timestamp'].dt.to_period('M').astype(str))
        .groupby(['station_id', 'station_name', 'period'], dropna=False)
        .agg(total_precip_mm=('precip_mm', 'sum'), wet_days=('precip_mm', lambda s: int((s > 0).sum())), mean_precip_mm=('precip_mm', 'mean'))
        .reset_index()
    )
    daily_annual_summary = (
        daily_records.assign(year=daily_records['timestamp'].dt.year)
        .groupby(['station_id', 'station_name', 'year'], dropna=False)
        .agg(total_precip_mm=('precip_mm', 'sum'), wet_days=('precip_mm', lambda s: int((s > 0).sum())), mean_precip_mm=('precip_mm', 'mean'))
        .reset_index()
    )
else:
    daily_monthly_summary = pd.DataFrame(columns=['station_id', 'station_name', 'period', 'total_precip_mm', 'wet_days', 'mean_precip_mm'])
    daily_annual_summary = pd.DataFrame(columns=['station_id', 'station_name', 'year', 'total_precip_mm', 'wet_days', 'mean_precip_mm'])

daily_monthly_summary.to_csv(DAILY_DIR / 'daily_monthly_summary.csv', index=False)
daily_annual_summary.to_csv(DAILY_DIR / 'daily_annual_summary.csv', index=False)

daily_wet_spell_events = build_precip_event_summary(daily_records, event_label='daily_wet_spell', gap=DAILY_WET_GAP, min_total_mm=MIN_EVENT_MM)
daily_wet_spell_events.to_csv(DAILY_EVENTS_DIR / 'daily_wet_spell_events.csv', index=False)

print(f'Daily records: {len(daily_records):,}')
print(f'Daily wet spells: {len(daily_wet_spell_events):,}')
display(daily_station_summary.sort_values('record_count', ascending=False).head(10))
display(daily_monthly_summary.head(10))
display(daily_annual_summary.head(10))
display(daily_wet_spell_events.head(10))


## 3. Hourly / sub-daily pipeline

Only stations and products with verified hourly or sub-hourly precipitation records enter this path.

If no hourly/subdaily source is verified, the notebook writes empty hourly outputs, preserves the daily analysis, and clearly reports that storm-event detection could not be executed.


In [ ]:
hourly_selection = availability_probe.loc[availability_probe['selected_for_event_analysis']].copy()
hourly_records = fetch_selected_series(hourly_selection, 'hourly', token) if not hourly_selection.empty else pd.DataFrame(columns=['station_id', 'station_name', 'source_station_id', 'source_dataset', 'data_type', 'analysis_kind', 'temporal_resolution', 'timestamp', 'precip_mm', 'year', 'month', 'day'])

hourly_records.to_csv(HOURLY_DIR / 'hourly_precipitation_records.csv', index=False)
hourly_station_summary = summarize_station_records(hourly_records)
hourly_station_summary.to_csv(HOURLY_DIR / 'hourly_station_summary.csv', index=False)

if hourly_records.empty:
    logger.warning('No verified hourly/subdaily precipitation records were found. Hourly storm-event detection skipped.')
    hourly_storm_events = pd.DataFrame(columns=['station_id', 'station_name', 'event_id', 'event_uid', 'event_kind', 'start', 'end', 'duration_hours', 'precip_mm', 'mean_intensity_mm_per_hr', 'n_records', 'wet_records', 'source_dataset', 'data_type', 'temporal_resolution'])
else:
    hourly_storm_events = build_precip_event_summary(hourly_records, event_label='hourly_storm_event', gap=HOURLY_DRY_GAP, min_total_mm=MIN_EVENT_MM)

hourly_storm_events.to_csv(HOURLY_EVENTS_DIR / 'hourly_storm_events.csv', index=False)

print(f'Hourly/subdaily records: {len(hourly_records):,}')
print(f'Hourly storm events: {len(hourly_storm_events):,}')
display(hourly_station_summary.sort_values('record_count', ascending=False).head(10))
display(hourly_storm_events.head(10))


## 4. Final availability table and summary

The final availability table merges the station audit with the full downloaded record summaries.

Summary rules:

- `daily_only`: daily precipitation was found, but no verified hourly/subdaily precipitation was found.
- `hourly_available`: hourly precipitation was verified and the station can be used for storm-event analysis.
- `subhourly_available`: sub-hourly precipitation was verified and the station can be used for storm-event analysis.
- `no_precip_found`: no useful precipitation records were found in the probed products.


In [ ]:
final_availability = build_final_availability_table(availability_probe, daily_records, hourly_records)
final_availability_gdf = make_station_gdf(final_availability, crs='EPSG:4326')
final_availability_local = final_availability_gdf.to_crs(LOCAL_CRS)

availability_csv = AVAILABILITY_DIR / 'station_precipitation_availability.csv'
availability_gpkg = AVAILABILITY_DIR / 'station_precipitation_availability.gpkg'
availability_local_gpkg = AVAILABILITY_DIR / 'station_precipitation_availability_projected.gpkg'

final_availability.to_csv(availability_csv, index=False)
final_availability_gdf.to_file(availability_gpkg, driver='GPKG', layer='station_precipitation_availability')
final_availability_local.to_file(availability_local_gpkg, driver='GPKG', layer='station_precipitation_availability_projected')

print(f'Availability table saved: {availability_csv}')
print(f'Availability GeoPackage saved: {availability_gpkg}')
print(f'Projected GeoPackage saved: {availability_local_gpkg}')

n_evaluated = len(final_availability)
n_daily_only = int((final_availability['station_type'] == 'daily_only').sum())
n_hourly = int((final_availability['station_type'] == 'hourly_available').sum())
n_subhourly = int((final_availability['station_type'] == 'subhourly_available').sum())
daily_used = final_availability.loc[final_availability['selected_for_daily_analysis'], 'station_id'].tolist()
event_used = final_availability.loc[final_availability['selected_for_event_analysis'], 'station_id'].tolist()

print(f'Stations evaluated: {n_evaluated:,}')
print(f'Daily-only stations: {n_daily_only:,}')
print(f'Hourly stations: {n_hourly:,}')
print(f'Subhourly stations: {n_subhourly:,}')
print('Daily analysis stations:', ', '.join(daily_used) if daily_used else 'none')
print('Event analysis stations:', ', '.join(event_used) if event_used else 'none')

if not event_used:
    print('Limitation: no verified hourly/subdaily precipitation source was found for the target stations in the probed NOAA/NCEI products. Daily analysis remains available as fallback, but it must be interpreted as daily wet-spell analysis, not storm-event analysis.')

summary_cols = ['station_id', 'station_type', 'source_dataset', 'data_type', 'temporal_resolution', 'start_date', 'end_date', 'record_count', 'selected_for_daily_analysis', 'selected_for_event_analysis', 'notes']
display(final_availability[summary_cols])

fig, ax = plt.subplots(figsize=(8, 4))
final_availability['station_type'].value_counts().sort_index().plot.bar(ax=ax)
ax.set_title('Stations by final precipitation availability type')
ax.set_xlabel('station_type')
ax.set_ylabel('count')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 10))
boroughs_plot = nyc_boroughs.to_crs(final_availability_gdf.crs)
boroughs_plot.boundary.plot(ax=ax, color='black', linewidth=1)
colors = {'daily_only': '#2563eb', 'hourly_available': '#d97706', 'subhourly_available': '#dc2626', 'no_precip_found': '#6b7280'}
for station_type, group in final_availability_gdf.groupby('station_type'):
    group.plot(ax=ax, color=colors.get(station_type, '#6b7280'), markersize=40, label=station_type)
set_map_extent(ax, boroughs_plot.unary_union, pad=0.05)
ax.set_title('Final precipitation station availability map')
ax.set_axis_off()
ax.legend(frameon=False, loc='lower left')
plt.tight_layout()
plt.show()

if event_used and not hourly_storm_events.empty:
    fig, ax = plt.subplots(figsize=(8, 4))
    hourly_storm_events['duration_hours'].plot.hist(bins=40, ax=ax)
    ax.set_title('Hourly storm-event duration')
    ax.set_xlabel('duration_hours')
    plt.tight_layout()
    plt.show()
